# N030 · 滑动窗口计数与恰好K

**目标：** 从以右端点结尾的合法起点数推导计数。

**先修：** N029, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** at_most；exactly差分；非负/单调边界；计数溢出意识。

**配套讲解来源：** [同名逐章意见](../../comment/030_window_counting_exactly_k.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题是**数子数组的个数**：给定一个数组 nums 和整数 k，问其中"恰好包含 k 种不同数字"的**连续子数组**一共有多少个。

具体例子：输入 `nums=[1,2,1,2,3]`、`k=2`，输出 `7`。为什么是 7？我们把所有恰好含 2 种数字的连续片段按左端点分组列出来（下标从 0 计）：

| # | 子数组 | 下标范围 | 用到的数字 |
|---|---|---|---|
| 1 | [1,2] | 0–1 | 1、2 |
| 2 | [1,2,1] | 0–2 | 1、2 |
| 3 | [1,2,1,2] | 0–3 | 1、2 |
| 4 | [2,1] | 1–2 | 1、2 |
| 5 | [2,1,2] | 1–3 | 1、2 |
| 6 | [1,2] | 2–3 | 1、2 |
| 7 | [2,3] | 3–4 | 2、3 |

注意三点：下标 0–4 的 [1,2,1,2,3] 含三种数字，不算；2–4 的 [1,2,3] 同样三种，不算；两段内容相同但位置不同的 [1,2]（0–1 与 2–3）要算两个。子数组按位置区分，这是计数题的通用约定。

这类题的麻烦在于"恰好 k 种"这个条件**不适合滑动窗口直接维护**：窗口扩大可能让种类从 k-1 变成 k（变合法），也可能从 k 变成 k+1（变非法），合法区间不连续。本章的核心技巧是绕开"恰好"，改算两个更好算的量再相减——这正是标题里"恰好K"三个字背后的玄机。

## 2. 基础知识：先读懂这些词

- **子数组（subarray）**：数组里一段连续的片段，由起点 l 和终点 r 唯一确定。注意它和"子序列"不同，子序列可以跳着选，子数组必须挨着。
- **"至多 k 种"（at_most_k）**：窗口内不同数字的个数不超过 k。这个条件对滑动窗口非常友好：种类超了就移出左端，移出只会减少种类或不变，收缩一定能恢复合法。
- **单调性（为什么"至多"好算）**：把窗口左端往右删元素，种类数只会变少或不变，永远不会变多。也就是说"合法的最左起点"只有一个临界位置 left，它左边全非法、右边全合法。这种"一边倒"的性质正是滑动窗口需要的。"恰好 k 种"没有这个性质，所以不直接算它。
- **差分技巧（exactly = atMost(k) − atMost(k−1)）**：一个巧妙的集合换算。所有"至多 k 种"的子数组构成一个大集合，"至多 k−1 种"是它的一个子集；把两个数相减，剩下的正好是"种类数恰好等于 k"的那部分，因为 k 种以下都被减掉了。写成公式就是 notebook 里的 `#(=k) = #(≤k) − #(≤k−1)`。
- **按右端点计数**：数子数组时我们按**右端点**分组统计。固定右端点 right，如果 left 是最小合法起点，那么起点取 left、left+1、…、right 都合法，共 `right−left+1` 个子数组。每个子数组的右端点唯一，所以这样数不会重复也不会遗漏。
- **defaultdict**：一个"查不存在的 key 也先给默认值 0"的字典。这里用它做频次表，省去"先判断 key 在不在"的样板代码。
  - 和 N029 的 Counter 对比：Counter 专门做计数（有 `counts[c]+=1` 之外的便捷方法），这里需要"计数之外还要随时删除 key"，defaultdict(int) 更顺手。
- **计数溢出意识**：子数组个数最多是 n(n+1)/2 个。n=10^5 时这个数约 50 亿，超出 32 位整数的上限；在 Python 里整数不溢出，但如果你用 C++/Java 写同类题，就需要 64 位整数（long long）来存答案。这是 “学习目标”部分 里"计数溢出意识"的含义。

## 3. 思路推导：从小例子开始

- **Step 1**：先写 `at_most_k_distinct(nums,k)`：维护一个窗口，使得窗口内不同数字种数永远 ≤ k。right 每右移一格，把新数字记入频次表；如果种类数超过 k，就不断把左端数字移出（它的计数减到 0 时要从字典里删掉这个 key，否则 `len(counts)` 虚高），直到种类回到 k 以内。
- **Step 2**：窗口合法后，累加 `right-left+1`。理由：当前窗口是以 right 结尾的最长合法片段，而它的任何"右半截"（起点更靠右的）种类只会更少，同样合法，所以恰有 `right-left+1` 个合法子数组以 right 结尾。
  - "右半截"用例子说：窗口 [1,2,1]（k=2，right=2，left=0）合法时，它的右半截 [2,1]、[1] 也都至多 2 种，三个一起被 `2−0+1=3` 这一步收进答案。
- **Step 3**：用差分得到"恰好 k 种"：`exactly_k_distinct(nums,k) = at_most_k_distinct(nums,k) - at_most_k_distinct(nums,k-1)`。直觉解释：至多 k 种的集合 = (种类 ≤ k−1 的集合) ∪ (种类恰好 k 的集合)，两块互不重叠，所以后者 = 前者 − 前一块。
- **Step 4**：处理边界：k<0 时不可能有任何子数组满足，直接返回 0；k=0 时非空子数组至少含 1 种数字，也是 0（代码里用 `k<=0: return 0` 一并挡住）。

手算演示（`a=[1,2,1,2,3]`，k=2，跑一遍 at_most）：

| right | 读入 | 收缩动作 | left | 窗口 | 新增个数 right−left+1 |
|---|---|---|---|---|---|
| 0 | 1 | 无 | 0 | [1] | 1 |
| 1 | 2 | 无 | 0 | [1,2] | 2 |
| 2 | 1 | 无 | 0 | [1,2,1] | 3 |
| 3 | 2 | 无 | 0 | [1,2,1,2] | 4 |
| 4 | 3 | 种类变 3，连移 1、2、1，删空两个 key | 3 | [2,3] | 2 |

at_most(2) = 1+2+3+4+2 = 12。同理可算出 at_most(1)=5（只有 5 个单元素子数组）、at_most(3)=15（全部 5×6/2=15 个子数组）。于是：

- 恰好 1 种 = 5 − 0 = 5
- 恰好 2 种 = 12 − 5 = **7**
- 恰好 3 种 = 15 − 12 = 3
- 恰好 4 种 = 15 − 15 = 0

这四行正是 notebook “运行示例”部分 用 `show_table` 打出来的表格（列是 k、至多k、至多k−1、恰好k），你可以先手算再运行对照。

再用 k=1 把"每个右端点新增几个"单独走一遍，帮你确认 `answer+=right-left+1` 这行到底在数什么：

| right | 读入 | 收缩后窗口 | left | 以 right 结尾的合法子数组 | 个数 |
|---|---|---|---|---|---|
| 0 | 1 | [1] | 0 | [1] | 1 |
| 1 | 2 | [2]（1 被移出） | 1 | [2] | 1 |
| 2 | 1 | [1]（2 被移出） | 2 | [1] | 1 |
| 3 | 2 | [2]（1 被移出） | 3 | [2] | 1 |
| 4 | 3 | [3]（2 被移出） | 4 | [3] | 1 |

五轮各新增 1 个，at_most(1)=5。每次窗口被压到只剩一个元素，是因为 [1,2] 相邻放不下（2 种），k=1 时任何两个相邻元素都构成 2 种，只能保留单元素窗口。

## 4. 核心代码：at_most_k_distinct

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def at_most_k_distinct(nums, k):
    if k < 0:
        return 0
    counts = defaultdict(int)
    left = answer = 0
    for right, x in enumerate(nums):
        counts[x] += 1
        while len(counts) > k:
            old = nums[left]
            counts[old] -= 1
            if counts[old] == 0:
                del counts[old]
            left += 1
        answer += right - left + 1
    return answer
```

### 逐段读懂代码

先看 `at_most_k_distinct`，对应实现如下：

（对应程序见下方分段实现与完整代码。）

- `if k<0: return 0`：k 是负数时连空窗口都非法，一步挡住非法参数，避免后面 `while len(counts)>k` 死循环（counts 的长度最小是 0，只有 k≥0 才可能满足）。
- `counts=defaultdict(int); left=answer=0`：counts 是频次表；left 是窗口左端；answer 是累计的子数组个数。
- `for right,x in enumerate(nums)`：right 扫全数组，x 是新进窗口的数字。
- `counts[x]+=1`：新数字记账。
- `while len(counts)>k:`：字典里 key 的个数就是窗口内不同数字的种数，超过 k 就非法。
- `old=nums[left]; counts[old]-=1`：把左端数字的计数减一。
- `if counts[old]==0: del counts[old]`：计数减到 0 必须把这个 key 从字典里删掉，因为 `len(counts)` 统计的是"种类数"，留着零计数的 key 会让种数虚报。
  - 举个例子：窗口 [1,2] 在 k=1 下收缩，若不移除计数归零的 key，counts 显示 2 种，while 会误以为还没合法而继续删，把窗口清空甚至死循环。
- `left+=1`：左端右移一格。整个 while 反复执行直到种数回到 k 以内。
  - 注意收缩的条件是 `len(counts)>k` 而不是 `>=k`：恰好 k 种是完全合法的，一步都不能多删。
- `answer+=right-left+1`：以 right 结尾的合法子数组个数，逐右端点累加。

再看 `exactly_k_distinct`，对应实现如下：

```python
def exactly_k_distinct(nums, k):
    if k <= 0:
        return 0
    return at_most_k_distinct(nums, k) - at_most_k_distinct(nums, k - 1)
```

- `if k<=0: return 0`：k=0 或负数时答案为 0。注意这里不能省：若 k=0，下面会去调 `at_most_k_distinct(nums,-1)`，虽然那个函数自己会返回 0，但表达式结果等于 at_most(0)−at_most(−1)，语义上"恰好 0 种的非空子数组"确实不存在，显式返回 0 更清楚也更快。
  - 对比两个函数的守卫：at_most 用 `k<0`（k=0 的 at_most 是良定义的，答案是 0，可以让主循环自己算出来）；exactly 用 `k<=0`（直接短路）。两处阈值不同是有意为之，想一想各自的理由。
- `return at_most_k_distinct(nums,k)-at_most_k_distinct(nums,k-1)`：差分公式本体。"至多 k 种"减去"至多 k−1 种"，剩下的就是"恰好 k 种"。

## 5. 分段实现：按顺序运行

**本章接口：** `at_most_k_distinct(nums, k)`、`exactly_k_distinct(nums, k)`

### 导入本章使用的库

In [1]:
from collections import defaultdict

### at_most_k_distinct

In [2]:
def at_most_k_distinct(nums, k):
    if k < 0:
        return 0
    counts = defaultdict(int)
    left = answer = 0
    for right, x in enumerate(nums):
        counts[x] += 1
        while len(counts) > k:
            old = nums[left]
            counts[old] -= 1
            if counts[old] == 0:
                del counts[old]
            left += 1
        answer += right - left + 1
    return answer

### exactly_k_distinct

In [3]:
def exactly_k_distinct(nums, k):
    if k <= 0:
        return 0
    return at_most_k_distinct(nums, k) - at_most_k_distinct(nums, k - 1)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,1,2,3]
show_table(['k','至多k','至多k−1','恰好k'],[(k,at_most_k_distinct(a,k),at_most_k_distinct(a,k-1),exactly_k_distinct(a,k)) for k in range(1,5)])

k,至多k,至多k−1,恰好k
1,5,0,5
2,12,5,7
3,15,12,3
4,15,15,0


## 7. 正确性、适用条件与复杂度

移除左端不会增加种类数，故最小合法left之后的所有起点都合法；更早起点均被证明非法。每个非空子数组按唯一右端点计数一次。

**代价：** 平均 O(n) 时间、O(k+1) 窗口计数空间；一般可写 O(u)。

### 展开理解

**为什么对**：关键在两件事。第一，对固定 right，左端删元素不会增加种数，所以合法起点是从某个 left 到 right 的连续一段；更靠左的起点在收缩循环里已经被证明非法（种数确实超了 k）。因此 `right-left+1` 恰好等于"以 right 结尾的合法子数组个数"，不多不少。拿 [1,2,1,2,3]、k=2、right=2 现场验证：窗口收缩后 left=0，以 right=2 结尾的合法子数组是起点取 0、1、2 的 [1,2,1]、[2,1]、[1]，正好 2−0+1=3 个；起点取 −1（即包含下标 0 之前）不存在，起点比 0 更小的也没有，数量精确吻合。第二，任何子数组都恰好有一个右端点，我们按右端点分组累加，同一个子数组不会被数两次，也不会有子数组漏网。差分公式的正确性用集合看：{种类 ≤ k} = {种类 ≤ k−1} ∪ {种类 = k}，两块没有公共元素，所以后者的个数等于两个数相减。

**复杂度**：at_most 里 right 走 n 步，left 全程最多也走 n 步（只进不退），所以两层循环摊下来总共 O(n) 次基本操作。exactly 调了两次 at_most，还是 O(n)。具体数字：n=10^5 时大约几十万次字典操作，毫秒级完成；对比暴力枚举所有 O(n^2) 个子数组再逐个数字典比对（约百亿次操作），完全不可同日而语。空间上频次表里最多存 k+1 个 key（窗口种数被压在 k 以内，收缩瞬间最多到 k+1），记作 O(k+1)；如果泛泛按"字符集/取值域大小 u"来写就是 O(u)。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解释：

- `assert exactly_k_distinct([1,2,1,2,3],2)==7`：正常值测试。就是第三节手算的那个例子，7 个子数组我们在第一节逐个列出来过，验证差分公式和窗口计数的主逻辑。
- `assert exactly_k_distinct([1,2],0)==0`：边界值测试。k=0 时任何非空子数组至少含 1 种数字，答案必须是 0，这条专门卡 `if k<=0` 那行守卫——如果守卫写漏或写反，这里会出错。
- 后面的大循环：用 `product([0,1,2],repeat=n)` 枚举长度 0~6、取值只有 0/1/2 的所有数组（3^0+...+3^6=1093 个），对每个数组和 k=0..3，用暴力参考和本章实现对拍。拆开看：
  - `ref=sum(len(set(a[l:r]))==k for l in range(n) for r in range(l+1,n+1))`：枚举每个非空子数组 [l,r)，用 `set` 数出不同值的个数，恰好等于 k 就计 1。这就是"恰好 k 种"的定义式暴力。
  - `r in range(l+1,n+1)` 从 l+1 起步，保证子数组非空——空子数组是 0 种，数进去了 ref 会系统性偏大。
  - 取值压到 0/1/2 三种、长度最多 6，是为了让"种类数触到 k 的各种边界组合"在小输入里都能出现：比如 k=3 只有在数组同时含 0、1、2 时才有非零答案，k=0 永远为 0（非空子数组至少 1 种），这两个边界都靠枚举兜底。

In [5]:
assert exactly_k_distinct([1, 2, 1, 2, 3], 2) == 7

assert exactly_k_distinct([1, 2], 0) == 0

from itertools import product

for n in range(7):
    for a in product([0, 1, 2], repeat=n):
        for k in range(4):
            ref = sum((len(set(a[l:r])) == k for l in range(n) for r in range(l + 1, n + 1)))
            assert exactly_k_distinct(a, k) == ref

print('N030: 所有本章断言通过')

N030: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推导exactly(K)=atMost(K)-atMost(K-1)。

**练习 2：** 解释含负数和限制为何可能失败。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（推导 exactly(K)=atMost(K)−atMost(K−1)）**：提示：从"每个子数组的种类数是唯一确定的"出发。把全部子数组按种类数分堆：0 种（不存在非空的）、1 种、2 种、…、K 种、…。atMost(K) 是前 K+1 堆之和，atMost(K−1) 是前 K 堆之和，相减剩下的恰是第 K 堆。写验证时可以取 `a=[1,1,2]`、K=2 手算：atMost(2)=6、atMost(1)=4（三个单元素加 [1,1]），exactly=6−4=2，即下标 1–2 的 [1,2] 和 0–2 的 [1,1,2]；再和暴力对拍。
  - 分堆验证表（`a=[1,1,2]`，共 6 个子数组）：
    - 1 种堆：[1]、[1]、[2]、[1,1] —— 4 个
    - 2 种堆：[1,2]（下标 1-2）、[1,1,2] —— 2 个
    - 对照：atMost(1)=4、atMost(2)=6、exactly(2)=6−4=2，与堆数一致。
- **练习 2（解释含负数和限制为何可能失败）**：提示：把约束从"种数 ≤ k"换成"子数组和 ≤ k"。非负数组里删掉左端元素和只会变小或不变，所以"和超标→收缩左端"一定能恢复合法；但含负数时，删掉一个负数反而让和变大，可能出现左端怎么删都回不到限额内、或最小合法 left 不存在（合法起点不连续）的情况。构造例子可试 `[-1, 4, -1]` 配不同 target 手算，观察"移出左端后窗口和上升"的瞬间，说明单调性断了、窗口结论不再成立：
  - 窗口 [-1, 4] 的和是 3；若限额是 2，把左端 −1 删掉后窗口变 [4]，和反升到 4——收缩让情况更糟。
  - 这种"越缩越糟"意味着 while 循环的终止前提（收缩单调改善合法性）不成立，窗口方法必须换掉（比如换 N032 的前缀哈希）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import defaultdict

def at_most_k_distinct(nums, k):
    if k < 0:
        return 0
    counts = defaultdict(int)
    left = answer = 0
    for right, x in enumerate(nums):
        counts[x] += 1
        while len(counts) > k:
            old = nums[left]
            counts[old] -= 1
            if counts[old] == 0:
                del counts[old]
            left += 1
        answer += right - left + 1
    return answer

def exactly_k_distinct(nums, k):
    if k <= 0:
        return 0
    return at_most_k_distinct(nums, k) - at_most_k_distinct(nums, k - 1)

# 示例与回归测试
assert exactly_k_distinct([1, 2, 1, 2, 3], 2) == 7

assert exactly_k_distinct([1, 2], 0) == 0

from itertools import product

for n in range(7):
    for a in product([0, 1, 2], repeat=n):
        for k in range(4):
            ref = sum((len(set(a[l:r])) == k for l in range(n) for r in range(l + 1, n + 1)))
            assert exactly_k_distinct(a, k) == ref

print('N030: 所有本章断言通过')

N030: 所有本章断言通过


## 11. 代表题与迁移

- **992. Subarrays with K Different Integers**：这题就是本章 `exactly_k_distinct` 的原题形态（求数组中恰好有 K 个不同整数的子数组个数），练的是"atMost 差分"这招。数组长度可达 2×10^4，恰好 K 种直接枚举是 O(n^2) 起步，差分 + 窗口线性解法轻松通过。
- **930. Binary Subarrays With Sum**：把"恰好 k 种"换成"元素和恰好为 S"的 01 数组计数题，同一套"前缀计数/差分"思想可以直接迁移（注意它用前缀和差而不是种数差）。01 数组里"和 ≤ S"同样满足"删左端不增"的单调性，所以 atMost(S) − atMost(S−1) 的套路原样可用——只是"种类数"换成了"1 的个数"。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。